# SED dependent reddening

[Galametz et al. (2017)](https://www.aanda.org/articles/aa/pdf/2017/02/aa29333-16.pdf) investigate the impact of SED dependent reddening. The Milky Way extinction is a function of the source SED. Because we don't know this ahead of time traditionally this was don eby 'dereddening' the measurements assuming a flat SED. Here we investigate the impact of SED dependent reddening on the low dust COSMOS field. This impact will be larger on areas with larger reddening in wide surveys.

Equation 2 in [Galametz et al. (2017)](https://www.aanda.org/articles/aa/pdf/2017/02/aa29333-16.pdf) provides a means to compute redeening dependent on the SED. The essential idea is that SEDs with a blue slope across the filter will be more reddened than SEDs with a red slope or flat spectra. In the paper they recommend computing the reddening for a fixed ebv value and assuming a linear relation. Here I simply mulitply the filter by the SED and then calculate the reddening as we would for the filter alone. I beleive this is mathematically equivalent.


In this notebook I want to use the LePHARE classes to compute a SED dependent reddening on library of SEDs and to investigate how it differs from a generic filter based approach.

In order to do this I exposed some functions in SED, Flt, PhotoZ and Mag.

If we decide the result is significant I don't think it would be too laborious to add a possibility to pass ebv to every onesource for PhotoZ and to add the magnitude/flux corrections during the fit. I would make this only available via the python run and try to do it in such a way that if switched off it had no im pact on time. At the most it might require an if clause inside the loop. Although by putting the function outside the loop the if could be avoided by setting the fit function outside and then calling whichever from inside.

If this is useful the bindings can be updated and the functions copied to the main code. In the first instance this will just be python side functions for investigating per sed per filter reddening and have no impact on the C++ code or speed.

In [ ]:
import lephare as lp
import numpy as np
from astropy.table import Table
from scipy.interpolate import interp1d
from matplotlib import pylab as plt
import struct
import timeit
import os

%matplotlib inline

## 1 Run prepare to give us access to library of SEDs

In [ ]:
config = lp.default_cosmos_config.copy()
bands = "ugrizy"

config.update(
    {
        "CAT_IN": os.path.join(lp.LEPHAREDIR, "examples/COSMOS_no_mw_correction.in"),
        # We look at a reduced filter list to demonstrate with ugrizy data
        "FILTER_LIST": f"cosmos/u_new.pb,hsc/gHSC.pb,hsc/rHSC.pb,hsc/iHSC.pb,hsc/zHSC.pb,hsc/yHSC.pb",
        # We use single values to deal with reduction in FILTER_LIST
        "ERR_SCALE": "0.02",  # Value from page 12 first paragraph in desprez 2023 diff fro phosphours
        "ERR_FACTOR": "1.5",  # Again from paper - diff for phosphorous - 2 lowered from 1.5 to 1. following chisquared dist
        "FILTER_CALIB": "0",
        "GLB_CONTEXT": np.sum(2 ** np.arange(len(bands))),
        "MABS_CONTEXT": np.sum(2 ** np.arange(len(bands))),
        # Reduced z grid for speed in demonstration
        "Z_STEP": ".02,0.,6.",
        # Reduced EM line dispersion for SPEED to demonstrate
        "EM_DISPERSION": "1.",
        ## THESE ARE THE VALUES THAT RELATE TO THE NEW REDDENING MODEL
        "EXT_ATMOS_CURVE": "NONE",
        "APPLY_MW_EXTINCTION": "GALAMETZ",  # "NO[DEF], CLASSIC, GALAMETZ"
        "EXT_MW_CURVE": "LMC_Fitzpatrick.dat",
        "MW_REFERENCE_MODEL": "sed/STAR/PICKLES/b5i.sed",  # The default - a B5 star
        # "MW_GLOBAL_EBV" : "0.016", # This can also be a file with id and ebv value corresponding to each source:
        "MW_EBV_FILE": os.path.join(
            lp.LEPHAREDIR, "examples/EBV_MW.in"
        ),  # FILE with IDs corresponding to the input and EBV values.
    }
)

In [ ]:
# Download the required data
lp.data_retrieval.get_auxiliary_data(
    keymap=config,
    # We need to have the additional files required for the Galametz method
    additional_files=[
        "examples/COSMOS_no_mw_correction.in",
        "examples/EBV_MW.in",
        "ext/LMC_Fitzpatrick.dat",
        "sed/STAR/PICKLES/b5i.sed",
    ],
)

In [ ]:
# Check the config values
config

In [ ]:
# Because we are dealing with some low level functionality we also need the keymap in the native LePHARE format
keymap = lp.all_types_to_keymap(config)

### Make the Model libraries

As for a typical run we run filter, sedtolib, and mag_mal to build the biraries. During thios run we now compute the extinction per band in addition to the band_pass_correction during mag_gal. When you initialize the PhotoZ object the band pass corrections are nbormalized to the reference model which is typically a B5 star.

In [ ]:
lp.prepare(keymap)

In [ ]:
# This loads the library of SEDs giving us access to the reddening per model
photz = lp.PhotoZ(keymap)

At this stage the EBV values per source have already been set as the EBV file was set at the input time. We can therefore inspect each value per source.

In [ ]:
sources = photz.read_photoz_sources()
photz.read_mw_ebv(sources)
sources[0].mw_ebv
mw_ebv_vals = np.array([s.mw_ebv for s in sources])
plt.hist(mw_ebv_vals, bins=30)
plt.xlabel("$E(B-V)$ [mag]")

## 2. Compute reddening 

We want to compute the baseline reddening as well as the per SED values for comparison. We do this using the original method. This function simply returns a global value for each filter. 

### Start with traditional values

Here we just get one value per band as a baseline "CLASSIC" run. These could be applied to the input catalogue as was done or applied to the models by setting APPLY_MW_EXTINCTION to CLASSIC.

In [ ]:
all_filters, _, _, baseline_albd = lp.classic_extinction_values(config)
baseline_albd

### Look at an example SED

We want to see how they vary over a filter and how the product differs from the pure filter and therefore impacts the reddening.

In [ ]:
# Pick one model to focus on
n_model = 20
# ex_sed=observe(photz, 5, mag)
keymap["t"] = lp.keyword("t", "Q")
qso_mag = lp.GalMag(keymap)
ex_sed = photz.fullLib[n_model]
print(f"Model {n_model} at redshift z={photz.zLib[n_model]} has its z=0 model at {ex_sed.index_z0}\n")
ex_sed.lamb_flux = photz.fullLib[ex_sed.index_z0].lamb_flux
ex_sed.generate_spectra(photz.zLib[n_model], 1)
ex_sed.data()

In [ ]:
# The first example is a QSO
ex_sed.is_qso()

We can then inspect the model dependent extinction values for any given model and compare them to the general values 

In [ ]:
# Afgter running prepare every SED in the library has the milky way extinction values in each band
ex_sed.milky_way_extinction

In [ ]:
bands

In [ ]:
fig, ax = plt.subplots()
filter_means = np.array([f.lambdaEff() for f in photz.allFilters])
filter_means
for n, b in enumerate(bands):
    x = np.full(len(photz.fullLib), filter_means[n])
    y = np.array([(s.milky_way_extinction[n] - baseline_albd[n]) for s in photz.fullLib])
    # plt.scatter(x,y)
    plt.violinplot(
        [y],
        positions=[filter_means[n]],
        widths=500,  # increase width manually
        bw_method=0.05,
    )
plt.xlabel("Filter mean wavelength [$\AA$]")
plt.ylabel("Difference in reddening")
# --- top axis ---
ax_top = ax.twiny()

# match limits so positions line up
ax_top.set_xlim(ax.get_xlim())

# your custom positions + labels
xpos = [filter_means[n] for n, b in enumerate(bands)]  # or whatever positions you want
xlabs = [b for b in bands]  # e.g. ['g', 'r', 'i', ...]

ax_top.set_xticks(xpos)
ax_top.set_xticklabels(xlabs)

### Plot the example SED

Check that the fluxes and SED are correct

In [ ]:
fluxes = np.array(photz.flux[n_model])  # (1/np.array(photz.flux[-n]))[::-1]
fluxes

In [ ]:
dwidth = np.array([f.dwidth for f in photz.allFilters])
dwidth

In [ ]:
from astropy.constants import c
import astropy.units as u

norm_band = 2
# ex_sed=observe(photz, 5, mag)
keymap["t"] = lp.keyword("t", "Q")
qso_mag = lp.GalMag(lp.all_types_to_keymap(keymap))

ex_sed = photz.fullLib[n_model]
print(f"Model {n_model} at redshift z={photz.zLib[n_model]} has its z=0 model at {ex_sed.index_z0}")
ex_sed.lamb_flux = photz.fullLib[ex_sed.index_z0].lamb_flux
ex_sed.generate_spectra(photz.zLib[n_model], 1)
ex_sed.data()

filter_means_temp = filter_means
fluxes = np.array(photz.flux[n_model])
fluxes2 = ex_sed.compute_fluxes(photz.allFilters)

lam = filter_means_temp * u.AA
Fnu = fluxes * u.erg / (u.s * u.cm**2 * u.Hz)

Flam = Fnu.to(u.erg / (u.s * u.cm**2 * u.AA), equivalencies=u.spectral_density(lam))
f = Flam  # fluxes*(u.erg/u.s/u.cm**2/u.Hz)*c.to(u.AA/u.s)/(filter_means_temp*u.AA)**2 #*dwidth
norm = np.interp(filter_means_temp, ex_sed.data()[0], ex_sed.data()[1])  # 2 should be r

plt.plot(ex_sed.data()[0], ex_sed.data()[1])
# plt.scatter(filter_means,comp_fluxes*norm/comp_fluxes[norm_band],c='r',marker='o')
plt.scatter(filter_means_temp, f * norm[norm_band] / f[norm_band], c="r", marker="o")
# Plot Lyman alpha to check redshift
plt.axvline(1215.67 * (1 + photz.zLib[n_model]), color="r", linestyle="--")
plt.title(f"z={photz.zLib[n_model]}")
plt.xlabel("Wavelength [$\AA$]")
plt.xlim([3000, 10000])
plt.ylim([0, 4.0 * np.max(f * norm[norm_band] / f[norm_band])])

### Define a Python function to look at the products. 
These products are performed by LEPHARE in c++ for speed. This funciton is merely for demonstration in the notebook to show how the product of the filter and the SED differs from the filter and impacts the reddening in a given band.



In [ ]:
def multiply_on_grids(x1, f1, x2, f2, x=None):
    """
    Interpolate two functions onto the same grid and multiply them elementwise.

    If no target grid `x` is provided, a grid spanning the union of `x1` and `x2`
    is automatically created with the same number of points as `x2`.

    Parameters
    ----------
    x1 : array_like
        Independent variable for the first function.
    f1 : array_like
        Function values corresponding to `x1`.
    x2 : array_like
        Independent variable for the second function.
    f2 : array_like
        Function values corresponding to `x2`.
    x : array_like, optional
        Grid on which to interpolate both functions. If None, an automatic grid
        spanning the min/max of `x1` and `x2` is used.

    Returns
    -------
    numpy.ndarray
        A 2D array with two rows: the first row is the grid `x`, and the second
        row is the product of the interpolated functions `f1 * f2`.

    Notes
    -----
    - Currently uses linear interpolation with `fill_value=0.0` outside the original ranges.
    - If the functions do not overlap we simply return the first function `f1`.
    """

    if x is None:
        print("Using user defined x grid")
        xmin = min(x1.min(), x2.min())
        xmax = max(x1.max(), x2.max())
        if xmax < xmin:
            # If no overlap return the input filter
            return np.array([x2, f2])
            return np.array([x1, f1])
        x = np.linspace(xmin, xmax, len(x2))
    f1i = interp1d(x1, f1, bounds_error=False, fill_value=0.0)(x)
    # f2 = gaussian_filter1d(f2, sigma=10)  # smooth the filter to avoid numerical issues with sharp features
    f2i = interp1d(x2, f2, bounds_error=False, fill_value=0.0)(x)
    prod = f1i * f2i
    norm = np.max(prod)
    norm = np.nanmax(prod)

    # If there is no overlap simply return f1
    if norm <= 0 or not np.isfinite(norm):
        x = x1
        y = f1
        # Allow it to be zero
        # y = prod
        # normalize to max of f1 to avoid numerical issues with very small values
        y = f1 / np.max(f1)
    else:
        y = prod / norm

    if not np.isfinite(np.array([x, y])).all():
        raise ValueError("Array contains NaN or infinite values")
    return np.array([x, y])

In [ ]:
# Normalise to this band
norm_band = 2
print(f"Model {n_model} at redshift z={photz.zLib[n_model]} has its z=0 model at {ex_sed.index_z0}")
ex_sed.lamb_flux = photz.fullLib[ex_sed.index_z0].lamb_flux
ex_sed.generate_spectra(photz.zLib[n_model], 1)
ex_sed.data()

model_filt_product = multiply_on_grids(
    all_filters[0].data()[0],
    all_filters[0].data()[1],
    ex_sed.data()[0],
    ex_sed.data()[1],
    x=all_filters[0].data()[0],
)
plt.plot(
    all_filters[0].data()[0], all_filters[0].data()[1] / np.max(all_filters[0].data()[1]), label="filter"
)
# plt.plot(oneFilt.data()[0],oneFilt.data()[1]/np.max(oneFilt.data()[1]))

plt.plot(
    ex_sed.data()[0],
    ex_sed.data()[1] / np.max(ex_sed.data()[1][(ex_sed.data()[0] < 4000) & (ex_sed.data()[0] > 3000)]),
    label="example sed",
)
plt.plot(model_filt_product[0], model_filt_product[1] / np.max(model_filt_product[1]), label="product")
plt.legend()
plt.xlim([np.min(all_filters[0].data()[0]), np.max(all_filters[0].data()[0])])
plt.ylim([0, 1.1])

plt.xlabel("Wavelength [Angstrom]")
plt.ylabel("Relative flux/transmission")

## 2.1 Use the python side function to get the extinction values

This runs prepare and loads the values per band. These are already computed by lephare.prepare and can also be accessed by interacting directly with the lephare.PhotoZ librarty of SEDs. We also have a helper function Python side to perform the full calculation and return the results:

In [ ]:
lp.compute_model_reddening?

In [ ]:
import time

start = time.time()
albd_lib = lp.compute_model_reddening(keymap)
end = time.time()

In [ ]:
print(
    f"Reddening calculation took {end-start:.2f} seconds or {(end-start)/albd_lib.shape[0]*1000:.3f} milliseconds per model"
)

In [ ]:
albd_lib.shape  # models by filters

### Look at distribution of extinction values

Note that the bluer bands have a peak at the low end where the SED has dropped out of the band due to redshifting and the code uses the reddest possible SED in such a case.

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib.T[n][~np.isnan(albd_lib.T[n]) & (albd_lib.T[n] > 0)]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data) / counts.max(),
        label=f"${b}$",
        color=colors[n],
        linewidth=2,
        alpha=0.5,
    )
    plt.axvline(baseline_albd[n], color=colors[n])
plt.legend(loc="upper left")
plt.ylabel("Relative density", fontsize=14)
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

In [ ]:
# Get masks to access the object types
qsos = np.array([s.is_qso() for s in photz.fullLib])
gals = np.array([s.is_gal() for s in photz.fullLib])
stars = np.array([s.is_star() for s in photz.fullLib])

In [ ]:
model_numbers = np.array([s.nummod for s in photz.fullLib])

In [ ]:
# This is the b5 star that determines the BPC normalisation
np.sum(stars & (model_numbers == 15))

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib[qsos].T[n][~np.isnan(albd_lib[qsos].T[n]) & (albd_lib[qsos].T[n] > 0)]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data) / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n], color=colors[n])
plt.legend(loc="upper right")
plt.ylabel("relative density")
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]")
plt.title("QSOs")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib[gals].T[n][(~np.isnan(albd_lib[gals].T[n])) & (albd_lib[gals].T[n] > 0)]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data) / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n], color=colors[n], linestyle="--")
plt.legend(loc="upper right")
plt.ylabel("relative density")
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]")
plt.title("Galaxies")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib[stars].T[n][(~np.isnan(albd_lib[stars].T[n])) & (albd_lib[stars].T[n] > 0)]
    # counts, bins = np.histogram(data, bins=bins)
    plt.hist(data, bins=bins, alpha=0.5, label=b, color=colors[n], linewidth=2)
    plt.axvline(baseline_albd[n], color=colors[n])
plt.legend(loc="upper right")
plt.ylabel("relative density")
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]")
plt.title("Stars")

## 2.2 Band pass correction

We also have a new function for getting the band pass correction based on hard coded B and V bands which are run on every object cpp side.

In [ ]:
lp.compute_band_pass_correction?

In [ ]:
t1 = time.time()
band_pass_correction = lp.compute_band_pass_correction(config)  #
t2 = time.time()

In [ ]:
print(
    f"BPC calculation took {t2-t1:.0f} seconds or {(t2-t1)/band_pass_correction.shape[0]*1000:.3f} milliseconds per model"
)

Note that that function reran prepare

In [ ]:
plt.hist(band_pass_correction, bins=30)
plt.xlabel("bpc$_{SED}$")

In [ ]:
elliptical = gals & (model_numbers < 8)

The following figures correspond to figure 3 in Galametz et al.

In [ ]:
plt.scatter(np.array(photz.zLib)[elliptical], band_pass_correction[elliptical])
plt.plot([0, 2], [1, 1])
plt.xlim([0, 2])
plt.ylim([0.8, 1.2])
plt.xlabel("z")
plt.ylabel("bpc$_{SED}$")
plt.title("Ellipticals")

In [ ]:
plt.scatter(np.array(photz.zLib)[elliptical], band_pass_correction[elliptical], s=0.2)
plt.plot([0, 6], [1, 1])
plt.xlim([0, 6])
plt.ylim([0.2, 2])
plt.xlabel("z")
plt.ylabel("bpc$_{SED}$")
plt.title("Ellipticals")

In [ ]:
plt.scatter(np.array(photz.zLib), band_pass_correction, s=0.2)
plt.plot([0, 6], [1, 1])
plt.xlim([0, 6])
plt.ylim([0.2, 2])
plt.xlabel("z")
plt.ylabel("bpc$_{SED}$")
plt.title("All models")

In [ ]:
albd_lib.shape, band_pass_correction.shape

### Check the redshift dependence of the values

In [ ]:
# We will look at redshifts to 2, 3 and the limit for the run at 6
red_2 = np.array(photz.zLib) < 2
red_3 = np.array(photz.zLib) < 3

We adopt an example E(B-V) of 0.1 to show the impact of the band pass correction as was done in Figure 8 of Galametz et al. (2017)

In [ ]:
galebv_ex = 0.1

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 0.6]
bin_width = 0.005
bins = np.arange(range[0], range[1] + bin_width, bin_width)

m = gals & red_2
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="-.")
plt.legend(loc="upper right")
plt.ylabel("# models per 0.005 mag")
plt.xlabel("Extinction $A_X$ [mag]")
plt.title("Galaxies at z<2 without bpc")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 0.6]
bin_width = 0.005
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 0.1
m = gals & red_2
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="-.")
plt.legend(loc="upper right")
plt.ylabel("# models per 0.005 mag")
plt.xlabel("Extinction $A_X$ [mag]")
plt.title("Galaxies at z<2")

Or alternatively use an artificial E(B-V) of 1 to plot the values scaled by the E(B-V) of the dust map

In [ ]:
galebv_ex = 1.0

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 1.0
m = gals & red_3
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
print("Galaxies to z=3")
plt.legend(loc="upper right", fontsize=8)
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_X/E(B-V)$ [mag]")
plt.title("Galaxy templates $z<3$")
plt.xlim(range)

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)

m = qsos & red_3
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
plt.legend(loc="upper right", fontsize=8)
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_{X}/E$(B-V) [mag]")
plt.title("QSO templates $z<3$")
plt.xlim(range)
print("QSO")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 1.0
m = qsos
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
plt.legend(loc="upper right", fontsize=8)
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_{X}/E$(B-V) [mag]")
plt.title("QSO templates $z<6$")
plt.xlim(range)
print("QSO")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 1.0
m = gals
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
plt.legend(loc="upper right")
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_{X}/E$(B-V) [mag]")
plt.title("Galaxy templates $z<6$")
plt.xlim(range)

### Look at some individual models and how the BPC changes with redshift

This is equivalent to figure 3 in Galametz et al. (2017)

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
gal_models = [1, 13]
gal_names = ["Ell", "Sc"]

qso_models = [7, 28]
qso_names = ["Seyfert 1.8", "QSO"]
galebv_ex = 0.1
band_number = 1
# Lets just look at objects with intrinsic ebv =0 for clarity
model_ebvs = np.array([m.ebv for m in photz.fullLib])
range = [0, 6]
for n, g in enumerate(gal_models):
    # print(g)
    mod_mask = np.array([s.nummod == g for s in photz.fullLib])
    mod_mask &= np.array([s.is_gal() for s in photz.fullLib])
    mod_mask &= model_ebvs == 0.0
    mod_reds = np.array(photz.zLib)[mod_mask]
    # data=albd_lib[mod_mask].T[band_number]
    # data=data*galebv_ex*band_pass_correction[mod_mask]
    data = band_pass_correction[mod_mask]
    plt.plot(mod_reds, data, label=gal_names[n])

for n, q in enumerate(qso_models):
    # print(q)
    mod_mask = np.array([s.nummod == q for s in photz.fullLib])
    mod_mask &= np.array([s.is_qso() for s in photz.fullLib])
    mod_mask &= model_ebvs == 0.0
    mod_reds = np.array(photz.zLib)[mod_mask]
    # data=albd_lib[mod_mask].T[band_number]
    # data=data*galebv_ex*band_pass_correction[mod_mask]
    data = band_pass_correction[mod_mask]
    plt.plot(mod_reds, data, label=qso_names[n])

# plt.plot(range,[baseline_albd[band_number],baseline_albd[band_number]],c='k',linestyle='--')
plt.plot(range, [1, 1], c="k", linestyle="--")
plt.xlim(range)
plt.xlabel("Redshift")
plt.ylabel("band pass correction")
plt.legend(fontsize=8)

## 3. Compute photoz and compare

We want to look at some actual outputs redshifts to see how it impacts results

In [ ]:
# Most of the other example use "dereddened" fluxes to account for redening using the traditional method.
# Here we need to use the raw observations
input_table = Table.read(os.path.join(lp.LEPHAREDIR, "examples/COSMOS_no_mw_correction.in"), format="ascii")

In [ ]:
cols = [input_table.colnames[0]] + input_table.colnames[3:15] + input_table.colnames[-3:]
input_table = input_table[cols]

In [ ]:
input_table[:5]

In [ ]:
len(input_table)

In [ ]:
### Make a reddened table as it was done to compare the impact
# input_table=Table.read("./data/COSMOS.fits")
# We are just using the ugrizy cols here
# cols = [input_table.colnames[0]]+ input_table.colnames[3:15] + input_table.colnames[-3:]
# input_table[cols][:5]
# ebv=0.016
# red_table=input_table[cols].copy()
# for n,b in enumerate('ugrizy'):
#     f_col = red_table.colnames[2*n+1]
#     ferr_col = red_table.colnames[2*n+2]
#     # correct for changes
#     red_table[f_col]/=10**(ebv*baseline_albd[n]/2.5)
#     red_table[ferr_col]/=10**(ebv*baseline_albd[n]/2.5)
# red_table.write('./data/COSMOS_not_derredened.fits', overwrite=True)

## run process with the reddening per model

We do this with the reddened measurements in addition to without and using the reddened models instead

In [ ]:
ebv = np.array(Table.read("./data/EBV_MW.in", format="ascii")["col2"])
ebv

In [ ]:
lp.prepare(config)
t1 = time.time()
out_galametz, _ = lp.process(config, input_table)
t2 = time.time()
# Run again using the classic method for comparison
lp.prepare({**config, "APPLY_MW_EXTINCTION": "CLASSIC"})
t3 = time.time()
out_classic, _ = lp.process({**config, "APPLY_MW_EXTINCTION": "CLASSIC"}, input_table)
t4 = time.time()

In [ ]:
print(f"Time per object with GALAMETZ reddening is a factor  {(t4-t3)/(t2-t1):.2f} larger")

In [ ]:
# How many results changed by more than 0.01?
zspec = out_classic["ZSPEC"]
z1 = out_classic["Z_BEST"]
z2 = out_galametz["Z_BEST"]
np.sum(np.abs(z1 - z2) > 0.01) / len(z1)

In [ ]:
m = np.abs(z1 - z2) > 0.001
plt.plot([0, 2], [0, 2], c="r")
plt.scatter(z1[m], z2[m], s=1)
plt.xlabel("z (Traditional method)")
plt.ylabel("z (Galametz)")
plt.xlim([0, 2])
plt.ylim([0, 2])

### How does the difference compare to general comparison with spectroscopic redshift

In [ ]:
plt.plot([0, 6], [0, 6], c="r")
plt.scatter(zspec[m], z1[m], s=1)
# plt.plot([0,2],[0,2])
plt.ylabel("z (Traditional method)")
plt.xlabel("z (Spec)")
plt.xlim([0, 6])
plt.ylim([0, 6])

In [ ]:
plt.plot([0, 6], [0, 6], c="r")
plt.scatter(zspec[m], z2[m], s=1)
# plt.plot([0,2],[0,2])
plt.ylabel("z (Galametz)")
plt.xlabel("z (Spec)")
plt.xlim([0, 6])
plt.ylim([0, 6])

### How many models changed as a result of the new method?

In [ ]:
same_gal = out_classic["MOD_BEST"] == out_galametz["MOD_BEST"]
same_qso = out_classic["MOD_QSO"] == out_galametz["MOD_QSO"]

In [ ]:
np.sum(~same_gal) / len(same_gal)

In [ ]:
np.sum(~same_qso) / len(same_gal)

In [ ]:
plt.scatter(z1, z2 - z1, s=5)
plt
plt.xlim([0, 6])
plt.ylim([-0.05, 0.05])
plt.plot([0, 6], [0, 0], c="r")
plt.xlabel("Z_BEST (Traditional)")
plt.ylabel("$\Delta$ Z_BEST (Galametz- Traditional)")

plt.title("Galaxies")

### Compare stats for the traditional method and the galaxy sample

In [ ]:
def sigma_nmad(z1, z2):
    maskPos = (z1 > 0.02) & (z2 > 0)
    delz = (z2 - z1) / (1 + z1)
    bias = np.nanmedian(delz[maskPos])
    med_delz = np.nanmedian(z2[maskPos] - z1[maskPos])
    delz_unbiased = (z2 - z1 - med_delz) / (1 + z1)
    outlier_frac = np.sum(np.abs(delz[maskPos]) > 0.15) / np.sum(maskPos)
    sigma_nmad_unbiased = 1.48 * np.nanmedian(np.abs(delz_unbiased[maskPos]))

    return {"sigma_nmad_unbiased": sigma_nmad_unbiased, "outlier_frac": outlier_frac, "bias": bias}

In [ ]:
# Classic
sigma_nmad(zspec, z1)

In [ ]:
# Galametz
sigma_nmad(zspec, z2)

Stats for the Galametz method and the galaxy sample

How long does the model reddening take?

In [ ]:
id, flux, flux_err, context, zspec, string_data = lp.table_to_data(config, input_table)
i = 0
one_obj = lp.onesource(i, photz.gridz)
one_obj.readsource(str(id[i]), flux[i], flux_err[i], context[i], zspec[i], str(string_data[i]))
photz.prep_data(one_obj)

In [ ]:
%timeit redened_flux=one_obj.redden_flux(photz.flux,albd_lib)

How does that compare to muliplying using numpy?

In [ ]:
%timeit red_num=photz.flux/10**(albd_lib/2.5)